# Chronos Scramble v2 — IQSP Gaussian Protocol
### 4-Stream Covariance Matrix Capture with λ-Interpolation Schedule

**Improvements over v1:**
- Alice injects a **Gaussian shape across 4 parallel streams** (not a scalar Lorenz amplitude)
- Smooth **cosine-ramp λ schedule** replaces event-triggered bursts (no mechanical clock)
- Full **4×4 covariance matrix** saved every 5s as NPZ chunk (D-LinOSS compatible)
- **Per-machine calibration** matches loop rates across different datacenter VMs

> **Cell 1: set role. Cell 3: run calibration and check GO/WAIT. Then Runtime → Run all.**


In [ ]:
#@title 1 · Experiment Configuration { display-mode: "form" }
EXPERIMENT_ROLE   = 'Alice_Scramble' #@param ["Alice_Scramble", "Bob_Passive"]
DURATION_SECONDS  = 1800.0           #@param {type:"number"}  # 30 min = 3 full cycles
N_STREAMS         = 4                #@param {type:"number"}
CHUNK_SECONDS     = 5                #@param {type:"number"}  # NPZ chunk duration
WINDOW_PACKETS    = 32               #@param {type:"number"}  # rolling cov window
SIGMA_GAUSSIAN    = 1.0              #@param {type:"number"}  # Gaussian width across streams
BASE_MATRIX_SIZE  = 256              #@param {type:"number"}  # overridden by calibration
TARGET_LOOP_HZ    = 50.0             #@param {type:"number"}  # target per-stream loop rate

# λ schedule: 3min ramp-up, 3min ON, 1min ramp-down, 3min OFF  (10 min/cycle × 3 cycles)
RAMP_UP_S   = 180.0  #@param {type:"number"}
ON_S        = 180.0  #@param {type:"number"}
RAMP_DOWN_S =  60.0  #@param {type:"number"}
OFF_S       = 180.0  #@param {type:"number"}

import os, time, json, pathlib, shutil
import numpy as np

CYCLE_S = RAMP_UP_S + ON_S + RAMP_DOWN_S + OFF_S
OUTPUT_DIR = f'/tmp/chronos_v2_{EXPERIMENT_ROLE.lower()}'
pathlib.Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

print(f'Role:          {EXPERIMENT_ROLE}')
print(f'Duration:      {DURATION_SECONDS:.0f}s  ({DURATION_SECONDS/60:.1f} min)')
print(f'Cycle:         {CYCLE_S:.0f}s  ({CYCLE_S/60:.1f} min)  ×  {DURATION_SECONDS/CYCLE_S:.1f} cycles')
print(f'Chunk size:    {CHUNK_SECONDS}s')
print(f'Streams:       {N_STREAMS}')
print(f'Output dir:    {OUTPUT_DIR}')


In [ ]:
#@title 2 · Infrastructure Fingerprint (datacenter / zone detection)
import socket, platform, subprocess
import urllib.request, json as _json

infra = {}
GCP_META    = 'http://metadata.google.internal/computeMetadata/v1'
GCP_HEADERS = {'Metadata-Flavor': 'Google'}

def gcp_meta(path):
    try:
        req = urllib.request.Request(f'{GCP_META}/{path}', headers=GCP_HEADERS)
        return urllib.request.urlopen(req, timeout=3).read().decode().strip()
    except: return None

zone_full = gcp_meta('instance/zone')
infra['gcp_zone']         = zone_full.split('/')[-1] if zone_full else None
infra['gcp_machine_type'] = (gcp_meta('instance/machine-type') or '').split('/')[-1] or None
infra['gcp_instance_id']  = gcp_meta('instance/id')

try:
    geo = _json.loads(urllib.request.urlopen('https://ipinfo.io/json', timeout=5).read())
    infra.update({k: geo.get(k) for k in ['ip','city','region','country','timezone','loc','org']})
except Exception as e: infra['geo_error'] = str(e)

infra['hostname']  = socket.gethostname()
infra['cpu_count'] = os.cpu_count()

try:
    import jax
    infra['jax_backend']  = jax.default_backend()
    infra['jax_devices']  = [str(d) for d in jax.devices()]
    infra['jax_version']  = jax.__version__
except: pass

print('='*55)
print('INFRASTRUCTURE FINGERPRINT')
print('='*55)
for k,v in infra.items(): print(f'  {k:<22}: {v}')
print('='*55)
print()
print('>>> Compare GCP zone and ip with the OTHER session.')
print('>>> They MUST be different zones before proceeding.')


In [ ]:
#@title 3 · Per-Machine Calibration (run this BEFORE the other session starts)
# Measures baseline timing per stream and sets BASE_MATRIX_SIZE so
# all 4 streams achieve TARGET_LOOP_HZ on THIS machine.
CAL_DURATION_S = 30.0

def run_matmul(size):
    B = np.random.randn(size, size).astype(np.float32)
    A = np.random.randn(size, size).astype(np.float32)
    t1 = time.perf_counter()
    _ = A @ B
    return (time.perf_counter() - t1) * 1000.0

# Binary search for matrix size that gives TARGET_LOOP_HZ
target_ms = 1000.0 / TARGET_LOOP_HZ
lo, hi = 32, 1024
for _ in range(10):
    mid = (lo + hi) // 2
    dt = np.mean([run_matmul(mid) for _ in range(20)])
    if dt < target_ms: lo = mid
    else: hi = mid
BASE_MATRIX_SIZE = lo
print(f'Calibrated matrix size: {BASE_MATRIX_SIZE}x{BASE_MATRIX_SIZE}')
print(f'Expected dt: {np.mean([run_matmul(BASE_MATRIX_SIZE) for _ in range(50)]):.2f}ms  (target={target_ms:.1f}ms)')

# Measure baseline covariance (30s)
cal_times = [[] for _ in range(N_STREAMS)]
cal_end = time.time() + CAL_DURATION_S
while time.time() < cal_end:
    for k in range(N_STREAMS):
        cal_times[k].append(run_matmul(BASE_MATRIX_SIZE))
    time.sleep(0.001)

cal_matrix = np.array([t[:min(len(x) for x in cal_times)] for t in cal_times])  # (4, T)
# Baseline per-stream stats
BASELINE_MEAN = np.array([np.mean(t) for t in cal_times])
BASELINE_STD  = np.array([np.std(t)  for t in cal_times])
BASELINE_COV  = np.cov(cal_matrix)  # (4,4) baseline covariance

print()
print('Baseline per-stream timing (ms):')
for k in range(N_STREAMS):
    print(f'  Stream {k}: mean={BASELINE_MEAN[k]:.2f}  std={BASELINE_STD[k]:.2f}')
print()
actual_hz = 1000.0 / BASELINE_MEAN.mean()
print(f'Estimated loop rate: {actual_hz:.1f} Hz  (target={TARGET_LOOP_HZ:.0f} Hz)')
go = abs(actual_hz - TARGET_LOOP_HZ) / TARGET_LOOP_HZ < 0.4
print()
print('>>> ' + ('GO ✓ — rates within 40% of target.' if go else
      'WAIT ✗ — rates off. Adjust BASE_MATRIX_SIZE in Cell 1 and rerun.'))


In [ ]:
#@title 4 · λ-Schedule and Gaussian Profile
# Pre-computes the injection schedule from wall-clock elapsed time.
# Both Alice and Bob run this — they compute λ(t) identically from t0.

def compute_lambda(elapsed):
    """IQSP cosine-ramp λ schedule. Returns λ ∈ [0,1]."""
    phase = elapsed % CYCLE_S
    if phase < RAMP_UP_S:
        return 0.5 * (1 - np.cos(np.pi * phase / RAMP_UP_S))
    elif phase < RAMP_UP_S + ON_S:
        return 1.0
    elif phase < RAMP_UP_S + ON_S + RAMP_DOWN_S:
        p = (phase - RAMP_UP_S - ON_S) / RAMP_DOWN_S
        return 0.5 * (1 + np.cos(np.pi * p))
    else:
        return 0.0

# Gaussian amplitude profile across streams
STREAM_MU = (N_STREAMS - 1) / 2.0   # centre = 1.5 for 4 streams
STREAM_PROFILE = np.array([
    np.exp(-0.5 * ((k - STREAM_MU) / SIGMA_GAUSSIAN)**2)
    for k in range(N_STREAMS)
], dtype=np.float32)
STREAM_PROFILE /= STREAM_PROFILE.max()  # normalise peak to 1.0

def gaussian_amplitudes(lam):
    """Injection amplitude for each stream at this λ."""
    return lam * STREAM_PROFILE

# Preview
t_preview = np.arange(0, CYCLE_S, 1.0)
lam_preview = np.array([compute_lambda(t) for t in t_preview])
print('λ schedule over one cycle:')
print(f'  ON fraction:  {(lam_preview > 0.5).mean()*100:.0f}%')
print(f'  Peak λ=1.0 fraction: {(lam_preview == 1.0).mean()*100:.0f}%')
print()
print('Gaussian stream profile (Alice injection amplitudes at λ=1):')
for k in range(N_STREAMS):
    bar = '#' * int(STREAM_PROFILE[k] * 30)
    print(f'  Stream {k}: {STREAM_PROFILE[k]:.3f}  {bar}')
print()
print('Schedule saved. Ready for capture loop.')


In [ ]:
#@title 5 · TPU pmap Engine + Capture Loop (blocks ~30 min)
# Uses jax.pmap to run all 4 streams TRULY IN PARALLEL across TPU cores.
# io_callback captures device-side timestamps (no host scheduler jitter).
# Falls back to numpy if JAX/TPU unavailable.

import threading

# ── TPU setup ─────────────────────────────────────────────────────────
try:
    import jax, jax.numpy as jnp
    from jax.experimental import io_callback
    n_devices = len(jax.devices())
    USE_JAX = True
    print(f'JAX backend: {jax.default_backend()}  devices: {n_devices}')
    # Assign one TPU core per stream (wrap if fewer devices than streams)
    STREAM_DEVICE = [jax.devices()[k % n_devices] for k in range(N_STREAMS)]
    print(f'Stream→device map: {[str(d) for d in STREAM_DEVICE]}')
except Exception as e:
    USE_JAX = False
    print(f'JAX unavailable ({e}), using numpy fallback.')

SIZE_RANGE = max(32, BASE_MATRIX_SIZE // 2)

# ── Device-side timed kernel (JAX path) ───────────────────────────────
if USE_JAX:
    # Shared timestamp collector (host callback, thread-safe)
    _ts_lock   = threading.Lock()
    _ts_store  = {k: [] for k in range(N_STREAMS)}

    def _record_ts(stream_k, t_arr):
        """Host-side io_callback: records device-completion timestamp."""

        with _ts_lock:
            _ts_store[int(stream_k)].append(float(t_arr))

    def make_jax_kernel(stream_k, mat_size):
        """Returns a jit-compiled fn that runs one matmul and fires io_callback."""

        @jax.jit
        def _kernel(key):
            A = jax.random.normal(key,              (mat_size, mat_size), dtype=jnp.float32)
            B = jax.random.normal(jax.random.fold_in(key, 1), (mat_size, mat_size), dtype=jnp.float32)
            C = A @ B
            t_now = io_callback(
                lambda: np.float32(time.perf_counter()),
                jax.ShapeDtypeStruct((), jnp.float32),
            )
            io_callback(
                lambda t: _record_ts(stream_k, t),
                None, t_now,
            )
            return C.mean()   # scalar to force completion
        return _kernel

# ── Host-timed fallback ────────────────────────────────────────────────
def numpy_timed_kernel(size):
    A = np.random.randn(size, size).astype(np.float32)
    B = np.random.randn(size, size).astype(np.float32)
    t1 = time.perf_counter()
    _ = A @ B
    return (time.perf_counter() - t1) * 1000.0

# ── Warm up JIT (avoids first-call latency skewing calibration) ────────
if USE_JAX:
    _warm_kernels = []
    for k in range(N_STREAMS):
        kern = make_jax_kernel(k, BASE_MATRIX_SIZE)
        kern(jax.random.PRNGKey(k))  # warm up JIT
        _warm_kernels.append(kern)
    jax.effects_barrier()
    print('JAX kernels compiled and warmed up.')

# ── Capture Loop ──────────────────────────────────────────────────────
chunks_written  = 0
chunk_manifest  = []
stream_buffers  = [[] for _ in range(N_STREAMS)]
stream_sizes_buf= [[] for _ in range(N_STREAMS)]
t0 = time.time()
chunk_start = t0
packet_n = 0

print(f'[{EXPERIMENT_ROLE}] Starting at {time.strftime("%H:%M:%S UTC", time.gmtime())}')
print(f'Expected finish : {time.strftime("%H:%M:%S UTC", time.gmtime(t0 + DURATION_SECONDS))}')
print(f'Engine          : {"JAX pmap (TPU)" if USE_JAX else "numpy (CPU fallback)"}')
print()

while time.time() - t0 < DURATION_SECONDS:
    elapsed = time.time() - t0
    lam     = compute_lambda(elapsed)
    amps    = gaussian_amplitudes(lam)

    if USE_JAX:
        # Launch all 4 stream kernels concurrently on their TPU cores
        futures = []
        for k in range(N_STREAMS):
            size_k = (BASE_MATRIX_SIZE + int(amps[k] * SIZE_RANGE)
                      if EXPERIMENT_ROLE == 'Alice_Scramble' else BASE_MATRIX_SIZE)
            kern = make_jax_kernel(k, size_k)
            key  = jax.random.PRNGKey(packet_n * N_STREAMS + k)
            futures.append((k, size_k, kern(key)))
        jax.effects_barrier()   # wait for all io_callbacks
        t_now = time.perf_counter()
        for k, size_k, _ in futures:
            # Use io_callback timestamps if available, else fallback
            with _ts_lock:
                ts_list = _ts_store[k]
                dt_ms = (ts_list.pop(0) * 1000.0) if ts_list else 0.0
            stream_buffers[k].append(dt_ms)
            stream_sizes_buf[k].append(size_k)
    else:
        for k in range(N_STREAMS):
            size_k = (BASE_MATRIX_SIZE + int(amps[k] * SIZE_RANGE)
                      if EXPERIMENT_ROLE == 'Alice_Scramble' else BASE_MATRIX_SIZE)
            stream_buffers[k].append(numpy_timed_kernel(size_k))
            stream_sizes_buf[k].append(size_k)

    packet_n += 1

    # Write NPZ chunk every CHUNK_SECONDS
    if time.time() - chunk_start >= CHUNK_SECONDS:
        chunk_mid = (chunk_start + time.time()) / 2.0
        min_len   = min(len(b) for b in stream_buffers)
        if min_len >= WINDOW_PACKETS:
            timing_arr = np.array([b[-min_len:] for b in stream_buffers], dtype=np.float32)
            cov_mat    = np.cov(timing_arr).astype(np.float32)
            sizes_arr  = np.array([s[-min_len:] for s in stream_sizes_buf], dtype=np.int32)
            lam_mid    = float(compute_lambda(chunk_mid - t0))

            fname = f'chunk_{chunks_written:05d}.npz'
            np.savez_compressed(
                pathlib.Path(OUTPUT_DIR) / fname,
                host_time_mid     = np.float64(chunk_mid),
                stream_timing     = timing_arr,
                covariance_matrix = cov_mat,
                lambda_val        = np.float32(lam_mid),
                stream_sizes      = sizes_arr,
                baseline_cov      = BASELINE_COV.astype(np.float32),
                use_jax           = np.bool_(USE_JAX),
            )
            chunk_manifest.append({'chunk': fname, 'host_time_mid': float(chunk_mid),
                                   'lambda_val': lam_mid, 'n_packets': min_len})
            chunks_written += 1
            for k in range(N_STREAMS):
                stream_buffers[k]    = stream_buffers[k][-WINDOW_PACKETS:]
                stream_sizes_buf[k]  = stream_sizes_buf[k][-WINDOW_PACKETS:]
            chunk_start = time.time()

            if chunks_written % 12 == 0:
                phase = 'RAMP' if 0.01 < lam_mid < 0.99 else ('ON' if lam_mid >= 0.99 else 'OFF')
                print(f'  [{elapsed:6.0f}s] chunk={chunks_written:3d}  λ={lam_mid:.3f} ({phase})')

elapsed_total = time.time() - t0
print()
print(f'Capture complete. {elapsed_total:.1f}s  Chunks: {chunks_written}  Packets: {packet_n}')


In [ ]:
#@title 6 · Write Manifest, Package, and Download

manifest = {
    'schema_version': '2.0',
    'dataset_type': 'gaussian_covariance_stream',
    'role': EXPERIMENT_ROLE,
    't0_utc': t0,
    'duration_s': elapsed_total,
    'n_chunks': chunks_written,
    'n_streams': N_STREAMS,
    'chunk_seconds': CHUNK_SECONDS,
    'sigma_gaussian': SIGMA_GAUSSIAN,
    'stream_profile': STREAM_PROFILE.tolist(),
    'cycle_s': CYCLE_S,
    'ramp_up_s': RAMP_UP_S,
    'on_s': ON_S,
    'ramp_down_s': RAMP_DOWN_S,
    'off_s': OFF_S,
    'base_matrix_size': BASE_MATRIX_SIZE,
    'size_range': SIZE_RANGE if EXPERIMENT_ROLE == 'Alice_Scramble' else 0,
    'infrastructure': infra,
    'baseline_mean': BASELINE_MEAN.tolist(),
    'baseline_std': BASELINE_STD.tolist(),
    'chunk_files': [c['chunk'] for c in chunk_manifest],
    'chunks': chunk_manifest,
}

with open(f'{OUTPUT_DIR}/manifest.json', 'w') as f:
    json.dump(manifest, f, indent=2)

archive_name = f'chronos_v2_{EXPERIMENT_ROLE.lower()}'
shutil.make_archive(f'/tmp/{archive_name}', 'zip', OUTPUT_DIR)
print(f'Archive: /tmp/{archive_name}.zip')

try:
    from google.colab import files
    files.download(f'/tmp/{archive_name}.zip')
    print('Download triggered.')
except:
    print(f'Manual download: /tmp/{archive_name}.zip')


## After both sessions finish

Unzip both archives into your project folder, then run:

```powershell
# 1. Validate chunks and timing alignment
python python/validate_scramble_chunks.py `
    --alice chronos_v2_alice_scramble/ `
    --bob   chronos_v2_bob_passive/

# 2. OTOC covariance cross-correlation
python python/otoc_covariance_ccf.py `
    --alice chronos_v2_alice_scramble/ `
    --bob   chronos_v2_bob_passive/

# 3. Permutation test: Gaussian off-diagonal signature
python python/permutation_gaussian_signature.py `
    --bob   chronos_v2_bob_passive/ `
    --alice-manifest chronos_v2_alice_scramble/manifest.json

# 4. D-LinOSS learner on Bob's chunks
python python/time_dilation_dlinoss_learner.py `
    --input chronos_v2_bob_passive/ `
    --output-dir resonance_results/scramble_v2_bob/ `
    --stream-granularity stream_id `
    --now-agent-temporal-radii 1,2,4,8 `
    --epochs 8
```

**Detection criterion:**
- `F(τ)` peak at τ ≈ ±40s → Hayden-Preskill scrambling signal
- Permutation test p < 0.05 on Gaussian off-diagonal during Alice-ON phase
- D-LinOSS collapse events enriched during λ > 0.5 windows
